In [1]:
import os
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import time
import torchvision.models as models
from matplotlib import pyplot as plt
import optuna

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [3]:
image_transforms = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [4]:
dataset_path = r"C:\Users\DELL\Data-Science-Projects\fresh-harvest-project\FRUIT-16K"

dataset = datasets.ImageFolder(root=dataset_path, transform=image_transforms)
len(dataset)

16000

In [5]:
class_names = dataset.classes
class_names

['F_Banana',
 'F_Lemon',
 'F_Lulo',
 'F_Mango',
 'F_Orange',
 'F_Strawberry',
 'F_Tamarillo',
 'F_Tomato',
 'S_Banana',
 'S_Lemon',
 'S_Lulo',
 'S_Mango',
 'S_Orange',
 'S_Strawberry',
 'S_Tamarillo',
 'S_Tomato']

In [6]:
num_classes = len(dataset.classes)
num_classes

16

In [7]:
train_size = int(0.75*len(dataset))
val_size = len(dataset) - train_size

train_size, val_size

(12000, 4000)

In [8]:
from torch.utils.data import random_split

train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

In [9]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=True)

In [10]:
class FruitAndFreshClassifierCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.network = nn.Sequential(
            nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, stride=1, padding=1), # (16, 224, 224)
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2, padding=0), # (16, 112, 112),
            nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2, padding=0), # (32, 56, 56)
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2, padding=0), # (64, 28, 28),
            nn.Flatten(),
            nn.Linear(64*28*28, 512),
            nn.ReLU(),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        x = self.network(x)
        return x

In [11]:
# Define the objective function for Optuna
def objective(trial):
    # Suggest values for the hyperparameters
    lr = trial.suggest_float('lr', 1e-5, 1e-2, log=True)

    # Load the model
    model = FruitAndFreshClassifierCNN(num_classes=num_classes).to(device)

    # Define the loss function and optimizer
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    # Training loop (using fewer epochs for faster hyperparameter tuning)
    epochs = 3
    start = time.time()
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        for batch_num, (images, labels) in enumerate(train_loader):
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * images.size(0)

        epoch_loss = running_loss / len(train_loader.dataset)

        # Validation loop
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        accuracy = 100 * correct / total

        # Report intermediate result to Optuna
        trial.report(accuracy, epoch)

        # Handle pruning (if applicable)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    end = time.time()
    print(f"Execution time: {end - start} seconds")

    return accuracy

In [ ]:
# Create the study and optimize
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=20)

[I 2026-07-05 10:22:58,541] A new study created in memory with name: no-name-2347669f-bf2a-466f-9c5f-33d7fe6889b8


Execution time: 3161.036392211914 seconds


[I 2026-07-05 11:15:41,100] Trial 0 finished with value: 88.075 and parameters: {'lr': 0.0005066875301674773}. Best is trial 0 with value: 88.075.
[I 2026-07-05 11:30:56,284] Trial 1 finished with value: 57.125 and parameters: {'lr': 1.062993235490819e-05}. Best is trial 0 with value: 88.075.


Execution time: 913.4059691429138 seconds


[I 2026-07-05 11:44:19,080] Trial 2 finished with value: 6.25 and parameters: {'lr': 0.005711987727744034}. Best is trial 0 with value: 88.075.


...

[I 2026-07-05 12:07:32,586] Trial 4 finished with value: 87.875 and parameters: {'lr': 0.0009730167372874033}. Best is trial 3 with value: 88.925.


Execution time: 727.704820394516 seconds


[I 2026-07-05 12:11:56,278] Trial 5 pruned. 
[I 2026-07-05 12:16:29,509] Trial 6 pruned. 
[I 2026-07-05 12:27:55,326] Trial 7 finished with value: 91.625 and parameters: {'lr': 0.0006154808841058006}. Best is trial 7 with value: 91.625.


Execution time: 685.1993913650513 seconds


[I 2026-07-05 12:47:10,268] Trial 8 pruned. 
[I 2026-07-05 12:52:11,482] Trial 9 pruned. 
